# Stock Return Prediction — Meta-Learning Model Study

Compares three base models as backbones for a **Transformer Portfolio Allocator**:

| Model file | Description |
|---|---|
| `model.pth` | Original trained predictor |
| `model_new.pth` | Retrained predictor |
| `meta_model.pth` | **New** — MetaStockPredictor trained with Reptile meta-learning |

Improvements in `MetaStockPredictor`:
- Temporal self-attention (2-layer Transformer over the 30-day window)
- Multi-task head: return regression + direction classification (BCE)
- **Reptile** meta-learning across random temporal episodes → generalises across market regimes
- Fine-tuning phase with cosine LR schedule after Reptile

## 1. Imports & Configuration

In [14]:
import os, time, warnings, copy
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import Dataset, DataLoader, Subset, TensorDataset

warnings.filterwarnings("ignore")

DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
TRAIN_PATH   = "data/dataset_train.csv"
TEST_PATH    = "data/dataset_test.csv"
SEQ_LEN      = 30
NUM_FEATURES = 18
BATCH_SIZE   = 128

FEATURE_COLS = [
    "open", "high", "low", "close", "volume",
    "ret_5", "ret_30", "rel_ret_5", "rank_ret_5",
    "vol_30", "risk_adj_ret", "vol_z", "pv_signal",
    "dist_high", "z_price", "residual", "market_ret_5", "corr_market_30",
]

print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")


Device : cuda
PyTorch: 2.11.0+cu130


## 2. Dataset Classes

In [15]:
class StockDataset(Dataset):
    """Per-ticker time-series dataset for base-model training / evaluation."""

    def __init__(self, data_path, seq_len=SEQ_LEN, scaler=None):
        self.seq_len = seq_len
        df = pd.read_csv(data_path)
        df.sort_values(["ticker", "timestamp"], inplace=True)
        for col in FEATURE_COLS:
            if col not in df.columns:
                df[col] = 0.0
        df.replace([np.inf, -np.inf], 0, inplace=True)
        df.fillna(0, inplace=True)
        df["target"] = df.groupby("ticker")["return"].shift(-1)
        df.dropna(subset=["target"], inplace=True)

        features = df[FEATURE_COLS].values
        targets  = df["target"].values
        tickers  = df["ticker"].values

        if scaler is None:
            self.mean = np.nanmean(features, axis=0)
            raw_std   = np.nanstd(features, axis=0)
            self.std  = np.where(raw_std == 0, 1.0, raw_std)
        else:
            self.mean, self.std = scaler

        features = (features - self.mean) / (self.std + 1e-8)
        self.valid_indices = [
            i for i in range(len(features) - seq_len)
            if tickers[i] == tickers[i + seq_len - 1]
        ]
        self.x_data = features
        self.y_data = targets

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, i):
        idx = self.valid_indices[i]
        x = self.x_data[idx : idx + self.seq_len]
        y = self.y_data[idx + self.seq_len - 1]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


def get_dataloaders(train_path=TRAIN_PATH, val_path=TEST_PATH,
                    seq_len=SEQ_LEN, batch_size=BATCH_SIZE):
    train_ds = StockDataset(train_path, seq_len=seq_len)
    scaler   = (train_ds.mean, train_ds.std)
    val_ds   = StockDataset(val_path, seq_len=seq_len, scaler=scaler)
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True,  drop_last=True),
        DataLoader(val_ds,   batch_size=batch_size, shuffle=False),
    )

print("StockDataset defined.")


StockDataset defined.


In [16]:
class PortfolioDataset(Dataset):
    """Groups every trading day into (num_stocks, seq_len, features) samples."""

    def __init__(self, data_path, seq_len=SEQ_LEN):
        self.seq_len = seq_len
        print(f"Loading portfolio data from {data_path} ...")
        df = pd.read_csv(data_path)
        df.sort_values(["timestamp", "ticker"], inplace=True)

        self.tickers    = sorted(df["ticker"].unique())
        self.num_stocks = len(self.tickers)
        self.dates      = sorted(df["timestamp"].unique())
        self.num_dates  = len(self.dates)

        D, S, F = self.num_dates, self.num_stocks, NUM_FEATURES
        self.X_all = np.zeros((D, S, F))
        self.Y_all = np.zeros((D, S))

        t2i = {t: i for i, t in enumerate(self.tickers)}
        d2i = {d: i for i, d in enumerate(self.dates)}

        for _, row in df.iterrows():
            di = d2i[row["timestamp"]]
            si = t2i[row["ticker"]]
            self.X_all[di, si, :] = row[FEATURE_COLS].values
            if "return" in row:
                self.Y_all[di, si] = row["return"]

        self.X_all = np.nan_to_num(self.X_all, nan=0.0, posinf=0.0, neginf=0.0)
        self.Y_all = np.nan_to_num(self.Y_all, nan=0.0, posinf=0.0, neginf=0.0)
        self.Y_all[:-1] = self.Y_all[1:]
        self.Y_all[-1]  = 0.0
        print(f"  {self.num_stocks} stocks | {self.num_dates} dates | {D - seq_len} valid samples")

    def __len__(self):
        return self.num_dates - self.seq_len

    def __getitem__(self, idx):
        seq_x    = np.transpose(self.X_all[idx : idx + self.seq_len], (1, 0, 2))
        target_y = self.Y_all[idx + self.seq_len - 1]
        return (
            torch.tensor(seq_x,    dtype=torch.float32),
            torch.tensor(target_y, dtype=torch.float32),
        )

print("PortfolioDataset defined.")


PortfolioDataset defined.


## 3. Shared Architectures

In [17]:
class ResidualConvBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv1 = nn.Conv1d(ch, ch, 3, padding=1)
        self.bn1   = nn.BatchNorm1d(ch)
        self.conv2 = nn.Conv1d(ch, ch, 3, padding=1)
        self.bn2   = nn.BatchNorm1d(ch)
        self.drop  = nn.Dropout(0.2)

    def forward(self, x):
        res = x
        out = F.gelu(self.bn1(self.conv1(x)))
        out = self.drop(out)
        out = self.bn2(self.conv2(out))
        return F.gelu(out + res)


class MultiWindowFeatureExtractor(nn.Module):
    """Attribute names match saved model.pth / model_new.pth checkpoints."""

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.short_conv = nn.Conv1d(in_dim, out_dim, 3, padding=1)
        self.mid_conv   = nn.Conv1d(in_dim, out_dim, 5, padding=2)
        self.long_conv  = nn.Conv1d(in_dim, out_dim, 7, padding=3)
        self.batch_norm = nn.BatchNorm1d(out_dim * 3)
        self.do         = nn.Dropout(0.3)

    def forward(self, x):
        return self.do(F.gelu(self.batch_norm(
            torch.cat([self.short_conv(x), self.mid_conv(x), self.long_conv(x)], dim=1)
        )))


class StockPricePredictor(nn.Module):
    """Original base predictor — multi-scale Conv + Residual backbone."""

    def __init__(self, num_features=NUM_FEATURES, hidden_size=128):
        super().__init__()
        self.initial_norm  = nn.BatchNorm1d(num_features)
        self.feature_block = MultiWindowFeatureExtractor(num_features, hidden_size)
        self.compression   = nn.Conv1d(hidden_size * 3, hidden_size, 1)
        self.res_blocks    = nn.Sequential(
            ResidualConvBlock(hidden_size),
            ResidualConvBlock(hidden_size),
            ResidualConvBlock(hidden_size),
        )
        self.output_head = nn.Sequential(
            nn.Linear(hidden_size, 128), nn.GELU(), nn.BatchNorm1d(128), nn.Dropout(0.4),
            nn.Linear(128, 64),          nn.GELU(),                       nn.Dropout(0.4),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.initial_norm(x)
        x = self.feature_block(x)
        x = F.gelu(self.compression(x))
        x = self.res_blocks(x)
        pooled = (x.mean(dim=-1) + x.max(dim=-1).values) / 2
        return self.output_head(pooled).squeeze(-1)

n_params = sum(p.numel() for p in StockPricePredictor().parameters())
print(f"StockPricePredictor — {n_params:,} parameters")


StockPricePredictor — 407,333 parameters


In [18]:
class TransformerAllocator(nn.Module):
    """
    Wraps a frozen backbone (StockPricePredictor or MetaStockPredictor) with a
    cross-sectional Transformer that learns portfolio weights.
    Handles both single-value and tuple-output backbones.
    """

    def __init__(self, backbone, hidden_dim=64):
        super().__init__()
        self.base_model = backbone
        for p in self.base_model.parameters():
            p.requires_grad = False

        self.feature_proj = nn.Sequential(
            nn.Linear(NUM_FEATURES + 1, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        enc = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=4, dim_feedforward=hidden_dim * 2,
            batch_first=True, dropout=0.1,
        )
        self.transformer  = nn.TransformerEncoder(enc, num_layers=2)
        self.scoring_head = nn.Sequential(
            nn.Linear(hidden_dim, 32), nn.GELU(), nn.Linear(32, 1),
        )

    def forward(self, x):
        B, S, T, F = x.shape
        with torch.no_grad():
            raw    = self.base_model(x.view(B * S, T, F))
            scores = raw[0] if isinstance(raw, tuple) else raw   # MetaStockPredictor returns tuple
        scores   = torch.nan_to_num(scores, 0.0).view(B, S, 1)
        combined = torch.cat([x[:, :, -1, :], scores], dim=-1)
        h        = self.transformer(self.feature_proj(combined))
        return torch.softmax(self.scoring_head(h).squeeze(-1), dim=-1)


class SharpeLoss(nn.Module):
    def __init__(self, ann_factor=float(np.sqrt(252))):
        super().__init__()
        self.ann = ann_factor

    def forward(self, weights, returns):
        pnl = (weights * returns).sum(dim=1)
        return -(pnl.mean() / (pnl.std() + 1e-8)) * self.ann + 0.1 * weights.pow(2).mean()

print("TransformerAllocator & SharpeLoss defined.")


TransformerAllocator & SharpeLoss defined.


## 4. Meta-Learning Architecture

`MetaStockPredictor` improves on `StockPricePredictor` with:

1. **Temporal self-attention** — 2-layer Transformer encoder over the 30-day sequence captures long-range dependencies that 1D convolutions miss
2. **Multi-task heads** — simultaneously predicts *return magnitude* (MSE) and *direction* (BCE), giving richer gradients
3. **Trained with Reptile** — samples temporal episodes (random market windows), adapts inside each episode, then nudges global weights toward adapted weights → learns an initialisation that generalises across market regimes

In [19]:
class TemporalAttention(nn.Module):
    """2-layer Transformer encoder over the time dimension."""

    def __init__(self, d_model=128, nhead=4, dropout=0.1):
        super().__init__()
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=2)

    def forward(self, x):           # x: (B, T, d_model)
        return self.encoder(x)


class MetaStockPredictor(nn.Module):
    """
    Enhanced predictor:
      Conv backbone → Temporal Attention → Multi-task heads.
    Trained with Reptile meta-learning.

    forward() → (return_score, direction_logit)
    score(x)  → return_score only  (compatible with TransformerAllocator)
    """

    def __init__(self, num_features=NUM_FEATURES, hidden=128):
        super().__init__()
        self.initial_norm  = nn.BatchNorm1d(num_features)
        self.feature_block = MultiWindowFeatureExtractor(num_features, hidden)
        self.compression   = nn.Conv1d(hidden * 3, hidden, 1)
        self.res_blocks    = nn.Sequential(
            ResidualConvBlock(hidden),
            ResidualConvBlock(hidden),
            ResidualConvBlock(hidden),
        )
        self.temporal_attn = TemporalAttention(d_model=hidden)
        self.trunk = nn.Sequential(
            nn.Linear(hidden * 2, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.2),
        )
        self.return_head = nn.Linear(128, 1)
        self.dir_head    = nn.Linear(128, 1)

    def forward(self, x):
        # x: (B, seq_len, num_features)
        xc = x.transpose(1, 2)          # (B, F, T)
        xc = self.initial_norm(xc)
        xc = self.feature_block(xc)
        xc = F.gelu(self.compression(xc))
        xc = self.res_blocks(xc)        # (B, hidden, T)

        xa = xc.transpose(1, 2)         # (B, T, hidden)
        xa = self.temporal_attn(xa)     # (B, T, hidden)

        pooled = torch.cat([xa.mean(dim=1), xa.max(dim=1).values], dim=-1)
        h = self.trunk(pooled)
        return self.return_head(h).squeeze(-1), self.dir_head(h).squeeze(-1)

    def score(self, x):
        """Return-only score for portfolio allocation."""
        ret, _ = self.forward(x)
        return ret

n_meta = sum(p.numel() for p in MetaStockPredictor().parameters())
print(f"MetaStockPredictor — {n_meta:,} parameters")


MetaStockPredictor — 746,662 parameters


In [20]:
class ReptileEpisodeSampler:
    """
    Pre-builds all train sequences; supports fast temporal episode sampling.
    An 'episode' = a random contiguous window of trading days.
    """

    def __init__(self, data_path, seq_len=SEQ_LEN, batch_size=64):
        print(f"Building episode sampler from {data_path} ...")
        df = pd.read_csv(data_path)
        df.sort_values(["ticker", "timestamp"], inplace=True)
        for col in FEATURE_COLS:
            if col not in df.columns:
                df[col] = 0.0
        df.replace([np.inf, -np.inf], 0, inplace=True)
        df.fillna(0, inplace=True)
        df["target"] = df.groupby("ticker")["return"].shift(-1)
        df.dropna(subset=["target"], inplace=True)

        feats    = df[FEATURE_COLS].values.astype(np.float32)
        mean_    = feats.mean(axis=0)
        std_     = feats.std(axis=0)
        std_     = np.where(std_ == 0, 1.0, std_)
        df[FEATURE_COLS] = (feats - mean_) / (std_ + 1e-8)

        self.seq_len    = seq_len
        self.batch_size = batch_size

        tickers    = df["ticker"].values
        timestamps = df["timestamp"].values
        features   = df[FEATURE_COLS].values.astype(np.float32)
        targets    = df["target"].values.astype(np.float32)

        all_x, all_y, all_ts = [], [], []
        for i in range(len(features) - seq_len):
            if tickers[i] == tickers[i + seq_len - 1]:
                all_x.append(features[i : i + seq_len])
                all_y.append(targets[i + seq_len - 1])
                all_ts.append(timestamps[i + seq_len - 1])

        self.X         = np.array(all_x,  dtype=np.float32)
        self.Y         = np.array(all_y,  dtype=np.float32)
        self.TS        = np.array(all_ts)
        self.unique_ts = sorted(set(self.TS))
        print(f"  {len(self.X):,} sequences | {len(self.unique_ts)} unique timestamps")

    def sample_episode(self, window=120, support_frac=0.65):
        """Returns (support_loader, query_loader); either may be None if too few samples."""
        max_start = max(0, len(self.unique_ts) - window - 1)
        start     = np.random.randint(0, max_start + 1)
        ts_win    = self.unique_ts[start : start + window]
        split_at  = int(len(ts_win) * support_frac)
        sup_ts    = set(ts_win[:split_at])
        qry_ts    = set(ts_win[split_at:])

        def loader(ts_set):
            mask = np.isin(self.TS, list(ts_set))
            if mask.sum() < 4:
                return None
            ds = TensorDataset(
                torch.tensor(self.X[mask]),
                torch.tensor(self.Y[mask]),
            )
            return DataLoader(ds, batch_size=self.batch_size, shuffle=True, drop_last=False)

        return loader(sup_ts), loader(qry_ts)

print("ReptileEpisodeSampler defined.")


ReptileEpisodeSampler defined.


## 5. Meta-Model Training

Two-phase training pipeline:

**Phase 1 — Reptile meta-learning**
For each outer step: sample N temporal episodes → clone model → take K inner SGD steps on each episode's support set → update global model toward the mean of adapted weights.

**Phase 2 — Fine-tuning**
Standard training on the full dataset with cosine LR decay to stabilise the Reptile initialisation.

In [21]:
def multi_task_loss(ret_pred, dir_logit, target, alpha=0.5):
    """0.5 × MSE(return) + 0.5 × BCE(direction)."""
    mse = F.mse_loss(ret_pred, target)
    bce = F.binary_cross_entropy_with_logits(dir_logit, (target > 0).float())
    return alpha * mse + (1.0 - alpha) * bce


def train_meta_model(model, sampler, train_loader, val_loader, save_path,
                     # Reptile
                     n_outer=250, n_tasks=4, inner_steps=8,
                     inner_lr=5e-4, meta_step=0.1,
                     # Fine-tune
                     ft_epochs=40, ft_lr=5e-5):
    """
    Full training pipeline for MetaStockPredictor.
    Loads from save_path if it exists (skips all training).
    Returns (model, reptile_loss_history, (ft_train_hist, ft_val_hist)).
    """
    if os.path.exists(save_path):
        model.load_state_dict(torch.load(save_path, map_location=DEVICE, weights_only=True))
        print(f"  Loaded from {save_path} — skipping training.")
        return model, [], ([], [])

    model.to(DEVICE)

    # ── Phase 1: Reptile ──────────────────────────────────────────────────────
    print(f"\n── Phase 1: Reptile  ({n_outer} outer steps, {n_tasks} tasks, {inner_steps} inner steps)")
    reptile_hist = []

    for outer in range(n_outer):
        deltas  = defaultdict(lambda: torch.zeros(1))
        qlosses = []
        valid   = 0

        for _ in range(n_tasks):
            sup_ld, qry_ld = sampler.sample_episode()
            if sup_ld is None:
                continue

            clone   = copy.deepcopy(model)
            opt_in  = torch.optim.Adam(clone.parameters(), lr=inner_lr)

            # Inner adaptation on support set
            clone.train()
            for _ in range(inner_steps):
                for bx, by in sup_ld:
                    bx, by = bx.to(DEVICE), by.to(DEVICE)
                    opt_in.zero_grad()
                    ret_p, dir_p = clone(bx)
                    multi_task_loss(ret_p, dir_p, by).backward()
                    torch.nn.utils.clip_grad_norm_(clone.parameters(), 1.0)
                    opt_in.step()

            # Accumulate delta θ_adapted - θ
            clone_sd = dict(clone.named_parameters())
            for name, param in model.named_parameters():
                d = clone_sd[name].data - param.data
                if valid == 0:
                    deltas[name] = d.clone()
                else:
                    deltas[name] = deltas[name] + d
            valid += 1

            # Query evaluation
            clone.eval()
            if qry_ld is not None:
                with torch.no_grad():
                    ql = sum(
                        multi_task_loss(*(clone(bx.to(DEVICE)) + (by.to(DEVICE),))).item()
                        for bx, by in qry_ld
                    ) / max(len(qry_ld), 1)
                qlosses.append(ql)

        if valid > 0:
            with torch.no_grad():
                for name, param in model.named_parameters():
                    param.data += meta_step * (deltas[name] / valid)

        avg = float(np.mean(qlosses)) if qlosses else 0.0
        reptile_hist.append(avg)
        if (outer + 1) % 50 == 0 or outer == 0:
            print(f"    Step {outer+1:4d}/{n_outer} | Query loss: {avg:.4f}")

    # ── Phase 2: Fine-tune ────────────────────────────────────────────────────
    print(f"\n── Phase 2: Fine-tuning  ({ft_epochs} epochs, lr={ft_lr})")
    opt   = torch.optim.AdamW(model.parameters(), lr=ft_lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ft_epochs, eta_min=ft_lr / 10)
    ft_train_hist, ft_val_hist = [], []

    for ep in range(ft_epochs):
        model.train()
        t_sum = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            opt.zero_grad()
            ret_p, dir_p = model(bx)
            loss = multi_task_loss(ret_p, dir_p, by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            t_sum += loss.item()
        sched.step()

        model.eval()
        v_sum = 0.0
        with torch.no_grad():
            for bx, by in val_loader:
                bx, by = bx.to(DEVICE), by.to(DEVICE)
                ret_p, dir_p = model(bx)
                v_sum += multi_task_loss(ret_p, dir_p, by).item()

        ft_train_hist.append(t_sum / len(train_loader))
        ft_val_hist.append(v_sum / len(val_loader))

        if (ep + 1) % 10 == 0 or ep == 0:
            lr_now = sched.get_last_lr()[0]
            print(f"    Epoch {ep+1:3d}/{ft_epochs} | "
                  f"Train: {ft_train_hist[-1]:.4f} | Val: {ft_val_hist[-1]:.4f} | "
                  f"LR: {lr_now:.2e}")

    torch.save(model.state_dict(), save_path)
    print(f"\n  Saved → {save_path}")
    return model, reptile_hist, (ft_train_hist, ft_val_hist)


In [ ]:
print("Building episode sampler ...")
sampler = ReptileEpisodeSampler(TRAIN_PATH)

print("\nLoading train / val loaders for fine-tuning ...")
train_loader, val_loader = get_dataloaders()

meta_model = MetaStockPredictor().to(DEVICE)
print(f"\n{'='*60}")
print("Training MetaStockPredictor with Reptile meta-learning")
print(f"{'='*60}")
meta_model, reptile_hist, (ft_train_hist, ft_val_hist) = train_meta_model(
    meta_model, sampler, train_loader, val_loader, "meta_model.pth",
)


Building episode sampler ...
Building episode sampler from data/dataset_train.csv ...
  190,176 sequences | 6393 unique timestamps

Loading train / val loaders for fine-tuning ...

Training MetaStockPredictor with Reptile meta-learning

── Phase 1: Reptile  (250 outer steps, 4 tasks, 8 inner steps)
    Step    1/250 | Query loss: 0.9930
    Step   50/250 | Query loss: 0.7891
    Step  100/250 | Query loss: 0.7376


In [ ]:
# ── Training curves (only shown when trained this session) ───────────────────
has_reptile = len(reptile_hist) > 0
has_ft      = len(ft_train_hist) > 0

if has_reptile or has_ft:
    ncols  = int(has_reptile) + int(has_ft)
    fig, axes = plt.subplots(1, ncols, figsize=(7 * ncols, 4))
    if ncols == 1:
        axes = [axes]
    ax_idx = 0

    if has_reptile:
        ax = axes[ax_idx]; ax_idx += 1
        ax.plot(range(1, len(reptile_hist) + 1), reptile_hist, lw=1.5)
        ax.set_title("Reptile — Query Loss per Outer Step", fontsize=11)
        ax.set_xlabel("Outer Step"); ax.set_ylabel("Multi-Task Loss"); ax.grid(True, alpha=0.3)

    if has_ft:
        ax = axes[ax_idx]
        ep = range(1, len(ft_train_hist) + 1)
        ax.plot(ep, ft_train_hist, lw=1.5, label="Train")
        ax.plot(ep, ft_val_hist,   lw=1.5, label="Val")
        ax.set_title("Fine-Tune Loss (MSE + BCE)", fontsize=11)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Multi-Task Loss")
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

    plt.suptitle("MetaStockPredictor — Training Curves", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("meta_model.pth loaded from checkpoint — no training curves to show.")


## 6. Load All Base Models

In [ ]:
def load_base_model(path):
    m = StockPricePredictor().to(DEVICE)
    m.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
    m.eval()
    return m

def load_meta_model_weights(path):
    m = MetaStockPredictor().to(DEVICE)
    m.load_state_dict(torch.load(path, map_location=DEVICE, weights_only=True))
    m.eval()
    return m

model_old  = load_base_model("model.pth")
model_new  = load_base_model("model_new.pth")
model_meta = load_meta_model_weights("meta_model.pth")
print("model.pth      loaded ✓")
print("model_new.pth  loaded ✓")
print("meta_model.pth loaded ✓")


In [ ]:
# ── Directional accuracy on validation set ────────────────────────────────────
print("Loading validation data for directional-accuracy check ...")
_, da_val_loader = get_dataloaders()

def directional_accuracy(model, loader):
    model.eval()
    preds_all, targets_all = [], []
    with torch.no_grad():
        for bx, by in loader:
            raw = model(bx.to(DEVICE))
            pred = raw[0] if isinstance(raw, tuple) else raw
            preds_all.append(pred.cpu().numpy())
            targets_all.append(by.numpy())
    p = np.concatenate(preds_all).flatten()
    t = np.concatenate(targets_all).flatten()
    mask = t != 0
    return float((np.sign(p[mask]) == np.sign(t[mask])).mean() * 100)

da_old  = directional_accuracy(model_old,  da_val_loader)
da_new  = directional_accuracy(model_new,  da_val_loader)
da_meta = directional_accuracy(model_meta, da_val_loader)

print(f"\n{'Model':<22} {'Dir Acc':>10}")
print("-" * 34)
print(f"{'model.pth':<22} {da_old:>9.2f}%")
print(f"{'model_new.pth':<22} {da_new:>9.2f}%")
print(f"{'meta_model.pth':<22} {da_meta:>9.2f}%  ← new")
print(f"{'Δ (meta − old)':<22} {da_meta - da_old:>+9.2f}%")


## 7. Portfolio Dataset & Loaders

In [ ]:
PORTFOLIO_TRAIN_FRAC = 0.8
PORTFOLIO_BATCH      = 16

portfolio_dataset = PortfolioDataset(TRAIN_PATH)

total         = len(portfolio_dataset)
train_size    = int(total * PORTFOLIO_TRAIN_FRAC)
train_indices = list(range(train_size))
test_indices  = list(range(train_size, total))

portfolio_train_loader = DataLoader(
    Subset(portfolio_dataset, train_indices),
    batch_size=PORTFOLIO_BATCH, shuffle=True,
)
portfolio_test_loader = DataLoader(
    Subset(portfolio_dataset, test_indices),
    batch_size=PORTFOLIO_BATCH, shuffle=False,
)

print(f"Portfolio split — train: {len(train_indices)} days | test: {len(test_indices)} days")


## 8. Train Transformer Allocators

One `TransformerAllocator` per backbone, identical hyperparameters, for a fair comparison.
Existing checkpoints are loaded automatically.

In [ ]:
TX_EPOCHS = 500
TX_LR     = 1e-4

def _tx_step(allocator, bx, by, criterion, optimizer):
    bx, by = bx.to(DEVICE), by.to(DEVICE)
    optimizer.zero_grad()
    loss = criterion(allocator(bx), by)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(
        [p for p in allocator.parameters() if p.requires_grad], max_norm=1.0,
    )
    optimizer.step()
    return loss.item()


def train_transformer_allocator(backbone, save_path, train_loader, test_loader,
                                 epochs=TX_EPOCHS, lr=TX_LR):
    allocator = TransformerAllocator(backbone).to(DEVICE)
    if os.path.exists(save_path):
        allocator.load_state_dict(
            torch.load(save_path, map_location=DEVICE, weights_only=True)
        )
        print(f"  Loaded existing weights from {save_path} — skipping training.")
        return allocator, [], []

    criterion = SharpeLoss()
    optimizer = torch.optim.AdamW(
        [p for p in allocator.parameters() if p.requires_grad],
        lr=lr, weight_decay=1e-2,
    )
    train_losses, val_losses = [], []
    print(f"  Training {epochs} epochs  →  {save_path}")

    for epoch in range(epochs):
        allocator.train(); allocator.base_model.eval()
        t_loss = sum(
            _tx_step(allocator, bx, by, criterion, optimizer)
            for bx, by in train_loader
        ) / len(train_loader)

        allocator.eval()
        with torch.no_grad():
            v_loss = sum(
                criterion(allocator(bx.to(DEVICE)), by.to(DEVICE)).item()
                for bx, by in test_loader
            ) / len(test_loader)

        train_losses.append(t_loss); val_losses.append(v_loss)
        if (epoch + 1) == 1 or (epoch + 1) % 100 == 0:
            print(f"    Epoch {epoch+1:4d}/{epochs} | "
                  f"Train: {t_loss:+.4f} | Val: {v_loss:+.4f}")

    torch.save(allocator.state_dict(), save_path)
    print(f"  Saved → {save_path}")
    return allocator, train_losses, val_losses


In [ ]:
print("=" * 60)
print("TX — backbone: model.pth")
print("=" * 60)
tx_old, tx_old_tl, tx_old_vl = train_transformer_allocator(
    model_old, "tx_old.pth", portfolio_train_loader, portfolio_test_loader)

print()
print("=" * 60)
print("TX — backbone: model_new.pth")
print("=" * 60)
tx_new, tx_new_tl, tx_new_vl = train_transformer_allocator(
    model_new, "tx_new.pth", portfolio_train_loader, portfolio_test_loader)

print()
print("=" * 60)
print("TX — backbone: meta_model.pth")
print("=" * 60)
tx_meta, tx_meta_tl, tx_meta_vl = train_transformer_allocator(
    model_meta, "tx_meta.pth", portfolio_train_loader, portfolio_test_loader)


In [ ]:
# ── TX training loss curves (only when trained this session) ─────────────────
trained = [(lbl, tl, vl) for lbl, tl, vl in [
    ("TX(model.pth)",     tx_old_tl,  tx_old_vl),
    ("TX(model_new.pth)", tx_new_tl,  tx_new_vl),
    ("TX(meta)",          tx_meta_tl, tx_meta_vl),
] if tl]

if trained:
    fig, axes = plt.subplots(1, len(trained), figsize=(7 * len(trained), 4))
    if len(trained) == 1:
        axes = [axes]
    for ax, (lbl, tl, vl) in zip(axes, trained):
        ep = range(1, len(tl) + 1)
        ax.plot(ep, tl, lw=1.5, label="Train")
        ax.plot(ep, vl, lw=1.5, label="Val")
        ax.set_title(lbl, fontsize=11)
        ax.set_xlabel("Epoch"); ax.set_ylabel("Neg. Sharpe Loss")
        ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    plt.suptitle("Transformer Allocator — Sharpe Loss During Training",
                 fontsize=12, fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    print("All allocators loaded from checkpoints — no training curves.")


## 9. Out-of-Sample Evaluation

In [ ]:
def evaluate_allocator(allocator, label):
    allocator.eval(); allocator.base_model.eval()
    dates, port_rets, ew_rets = [], [], []

    with torch.no_grad():
        for idx in sorted(test_indices):
            d_idx = idx + portfolio_dataset.seq_len - 1
            date  = portfolio_dataset.dates[d_idx]
            seq_x, target_y = portfolio_dataset[idx]
            weights  = allocator(seq_x.unsqueeze(0).to(DEVICE)).squeeze(0).cpu().numpy()
            target_y = target_y.numpy()
            dates.append(date)
            port_rets.append(float(np.dot(weights, target_y)))
            ew_rets.append(float(target_y.mean()))

    return pd.DataFrame({
        "date":         pd.to_datetime(dates),
        label:          port_rets,
        "equal_weight": ew_rets,
    }).set_index("date").sort_index()


print("Evaluating TX(model.pth) ...")
res_old  = evaluate_allocator(tx_old,  "TX(model.pth)")
print("Evaluating TX(model_new.pth) ...")
res_new  = evaluate_allocator(tx_new,  "TX(model_new.pth)")
print("Evaluating TX(meta) ...")
res_meta = evaluate_allocator(tx_meta, "TX(meta)")

results = (
    res_old[["TX(model.pth)"]]
    .join(res_new[["TX(model_new.pth)"]])
    .join(res_meta[["TX(meta)"]])
    .join(res_old[["equal_weight"]])
)
print(f"\nEvaluation complete — {len(results)} test dates")
print(results.describe().round(5))


## 10. Comparison Visualisation

In [ ]:
cum = results.cumsum()          # cumulative PnL via sum (returns can exceed ±1)

COLS  = ["TX(model.pth)", "TX(model_new.pth)", "TX(meta)", "equal_weight"]
CLRS  = ["steelblue", "darkorange", "seagreen", "grey"]
LBLS  = ["TX (model.pth)", "TX (model_new.pth)", "TX (meta — Reptile)", "Equal Weight"]
LSTYS = ["-", "-", "-", "--"]

fig = plt.figure(figsize=(16, 14))
fig.suptitle(
    "Transformer Allocator — model.pth vs model_new.pth vs MetaStockPredictor (Reptile)",
    fontsize=13, fontweight="bold", y=0.998,
)
gs = gridspec.GridSpec(3, 2, hspace=0.48, wspace=0.32)

# 1. Cumulative PnL (full width)
ax1 = fig.add_subplot(gs[0, :])
for col, c, lbl, ls in zip(COLS, CLRS, LBLS, LSTYS):
    ax1.plot(cum.index, cum[col], lw=2.0, color=c, label=lbl, ls=ls, alpha=0.9)
ax1.axhline(0, color="grey", lw=0.8, ls=":")
ax1.set_title("Cumulative Out-of-Sample PnL (sum of daily returns)", fontsize=11)
ax1.set_xlabel("Date"); ax1.set_ylabel("Cumulative Return (sum)")
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

# 2. Daily returns
ax2 = fig.add_subplot(gs[1, 0])
for col, c, lbl in zip(COLS[:3], CLRS[:3], LBLS[:3]):
    ax2.plot(results.index, results[col], alpha=0.6, lw=0.9, color=c, label=lbl)
ax2.axhline(0, color="grey", lw=0.8, ls="--")
ax2.set_title("Daily Portfolio Returns", fontsize=11)
ax2.set_xlabel("Date"); ax2.set_ylabel("Daily Return")
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

# 3. Rolling 30-day Sharpe
ax3 = fig.add_subplot(gs[1, 1])
for col, c, lbl in zip(COLS[:3], CLRS[:3], LBLS[:3]):
    r  = results[col]
    rs = r.rolling(30).mean() / (r.rolling(30).std() + 1e-8) * np.sqrt(252)
    ax3.plot(rs.index, rs, lw=1.4, color=c, label=lbl)
ax3.axhline(0, color="grey", lw=0.8, ls="--")
ax3.set_title("Rolling 30-Day Annualised Sharpe Ratio", fontsize=11)
ax3.set_xlabel("Date"); ax3.set_ylabel("Sharpe"); ax3.legend(fontsize=8); ax3.grid(True, alpha=0.3)

# 4. Return distribution
ax4 = fig.add_subplot(gs[2, 0])
for col, c, lbl in zip(COLS, CLRS, LBLS):
    ax4.hist(results[col], bins=50, alpha=0.4, density=True, color=c, label=lbl)
ax4.set_title("Daily Return Distribution", fontsize=11)
ax4.set_xlabel("Daily Return"); ax4.set_ylabel("Density")
ax4.legend(fontsize=8); ax4.grid(True, alpha=0.3)

# 5. Drawdown (on cumulative PnL)
ax5 = fig.add_subplot(gs[2, 1])
for col, c, lbl in zip(COLS, CLRS, LBLS):
    cpnl = cum[col]
    dd   = cpnl - cpnl.cummax()
    ax5.fill_between(dd.index, dd, 0, alpha=0.25, color=c, label=lbl)
    ax5.plot(dd.index, dd, lw=0.8, color=c, alpha=0.7)
ax5.set_title("Drawdown (absolute, on cumulative PnL)", fontsize=11)
ax5.set_xlabel("Date"); ax5.set_ylabel("Drawdown")
ax5.legend(fontsize=8); ax5.grid(True, alpha=0.3)

plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → model_comparison.png")


## 11. Summary Metrics

In [ ]:
def compute_metrics(series, ann_factor=252):
    r        = np.asarray(series, dtype=float)
    cum_pnl  = np.cumsum(r)
    total    = float(cum_pnl[-1])
    ann_r    = float(r.mean() * ann_factor)
    ann_v    = float(r.std(ddof=1) * np.sqrt(ann_factor))
    sharpe   = ann_r / ann_v if ann_v > 0 else float("nan")
    roll_max = np.maximum.accumulate(cum_pnl)
    max_dd   = float((cum_pnl - roll_max).min())
    win_rate = float((r > 0).mean())
    return {
        "Total PnL (Σ)":     f"{total:+.4f}",
        "Ann. Return":        f"{ann_r:+.4f}",
        "Ann. Volatility":    f"{ann_v:.4f}",
        "Sharpe Ratio":       f"{sharpe:.3f}",
        "Max Drawdown":       f"{max_dd:.4f}",
        "Win Rate":           f"{win_rate * 100:.1f}%",
        "Avg Daily Return":   f"{r.mean():+.6f}",
    }

summary = pd.DataFrame({
    "TX(model.pth)":     compute_metrics(results["TX(model.pth)"]),
    "TX(model_new.pth)": compute_metrics(results["TX(model_new.pth)"]),
    "TX(meta — Reptile)":compute_metrics(results["TX(meta)"]),
    "Equal Weight":      compute_metrics(results["equal_weight"]),
})

print("\n" + "═" * 76)
print("   OUT-OF-SAMPLE PERFORMANCE SUMMARY")
print("═" * 76)
print(summary.to_string())
print("═" * 76)

# highlight winner per row
print("\nWinner per metric (excluding Equal Weight):")
for metric in summary.index:
    row = summary.loc[metric]
    try:
        vals = {k: float(str(v).replace("%","").replace("+","")) for k,v in row.items()
                if k != "Equal Weight"}
        winner = max(vals, key=vals.get)
        print(f"  {metric:<22} → {winner}")
    except Exception:
        pass
